# Temporal spectral analysis — sanity run
SCOTUS co-citation graph at several dates: spectrum, embedding, named extremes, subspace drift. Every section says what 'sensible' looks like.

In [ ]:
import sys, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = "."                 # <-- directory that contains lawgraph/
DB   = "graph.duckdb"      # <-- your file
sys.path.insert(0, ROOT)
from lawgraph.store import Store
from lawgraph.lens import select, build, Spectral, align, drift
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 70)
st = Store(DB, read_only=True); con = st.con

COURT = "scotus"
YEARS = [1900, 1920, 1940, 1960, 1980, 2000, 2020]
K     = 12                 # eigenvectors to keep
MODE  = "cocite"           # "cocite" | "sym" | "bibcoup"
MIN_DEG = 5

def names(ids):
    df = con.execute("SELECT id, case_name, year(date_filed) AS year FROM node_cluster WHERE id IN (SELECT unnest(?))",
                     [list(map(int, ids))]).df().set_index("id")
    return df.reindex(ids)

## 1. Build one slice and look at it
**Sensible:** tens of thousands of nodes, a few pruned, eigenvalues small and increasing, a visible gap somewhere in the first ten.

In [ ]:
t0 = time.time()
s = st.at(2000)
ids = select(s, "cluster", where=f"d.dst_id = '{COURT}' AND n.precedential_status = 'Published'",
             join="JOIN edge_cluster_in_docket_docket cd ON cd.src_id = n.id JOIN edge_docket_filed_in_court d ON d.src_id = cd.dst_id")
G = build(s, ("cluster", "cites", "cluster"), ids, mode=MODE, min_degree=MIN_DEG)
print(f"selected {len(ids):,} -> graph {G.n:,} nodes, {G.W.nnz//2:,} edges, pruned {G.meta}   ({time.time()-t0:.1f}s)")
emb = Spectral(k=K)(G)
print("eigenvalues:", emb.vals.round(4))
print("gap hint -> about", emb.gap(), "communities")
plt.figure(figsize=(6,3)); plt.plot(range(2, K+2), emb.vals, "o-"); plt.xlabel("index"); plt.ylabel("λ"); plt.title("bottom of the spectrum, 2000"); plt.show()
ipr = emb.localization()
print("localization (IPR) per eigenvector, ~%.4f = spread out, >0.05 = stuck on a few nodes:" % (1/G.n))
print(np.round(ipr, 4))


## 2. Do the eigenvectors read law?
**Sensible:** the two ends of the Fiedler vector are recognisably different areas of law (e.g. criminal procedure vs. commercial/federal-courts). If both ends look like random cases, raise `MIN_DEG` or switch `MODE`.

In [ ]:
deg = G.degree
for j in range(3):
    v = emb.vecs[:, j]
    lo, hi = np.argsort(v)[:8], np.argsort(-v)[:8]
    # among well-connected nodes only, so we see anchors rather than oddities
    busy = np.where(deg >= np.percentile(deg, 90))[0]
    lo_b, hi_b = busy[np.argsort(v[busy])[:8]], busy[np.argsort(-v[busy])[:8]]
    print(f"\n=== eigenvector {j+1}  (λ = {emb.vals[j]:.4f}) ===")
    print("  negative end:"); print(names(G.ids[lo_b])[["case_name", "year"]].to_string(header=False))
    print("  positive end:"); print(names(G.ids[hi_b])[["case_name", "year"]].to_string(header=False))

## 2b. More than three regions
K-means on the leading eigenvectors; set `N_COMM` to anything. Each group is described by its highest-degree members.
**Sensible:** each group is nameable; the three arms split into recognisable sub-areas rather than random mixtures.

In [ ]:
from scipy.cluster.vq import kmeans2
N_COMM = 7
X = emb.vecs[:, :N_COMM - 1]
X = X / np.linalg.norm(X, axis=1, keepdims=True)                 # spherical: direction matters, not distance from hub
_, lab = kmeans2(X, N_COMM, minit="++", seed=0)
deg = G.degree
for c in range(N_COMM):
    m = np.where(lab == c)[0]
    top = m[np.argsort(-deg[m])[:6]]
    nm = names(G.ids[top])
    print(f"--- community {c}: {len(m):,} cases, median year {int(np.median(names(G.ids[m]).year))}")
    for cid, r in nm.iterrows(): print(f"      {r.case_name[:60]}  {r.year}")
plt.figure(figsize=(6,6)); plt.scatter(emb.vecs[:,0], emb.vecs[:,1], c=lab, s=2, cmap="tab10", alpha=.6); plt.title(f"{N_COMM} spectral communities"); plt.show()


## 3. Known pairs: near or far in the embedding?
**Sensible:** cases from the same doctrine are closer than cases from different doctrines, by a clear margin.

In [ ]:
PAIRS = {  # (name fragment, year) pairs; adjust freely
    "same: Miranda / Terry":        (("Miranda v. Arizona", 1966), ("Terry v. Ohio", 1968)),
    "same: Twombly / Conley":       (("Bell Atlantic Corp. v. Twombly", 2007), ("Conley v. Gibson", 1957)),
    "same: Celotex / Anderson":     (("Celotex", 1986), ("Anderson v. Liberty Lobby", 1986)),
    "diff: Miranda / Chevron":      (("Miranda v. Arizona", 1966), ("Chevron", 1984)),
    "diff: Terry / Celotex":        (("Terry v. Ohio", 1968), ("Celotex", 1986)),
    "diff: Strickland / Erie":      (("Strickland v. Washington", 1984), ("Erie Railroad v. Tompkins", 1938)),
}
def cid(name, year):
    r = con.execute(f"SELECT id FROM node_cluster WHERE case_name ILIKE '%{name}%' AND year(date_filed) = {year} ORDER BY citation_count DESC NULLS LAST LIMIT 1").fetchall()
    return r[0][0] if r else None
V = emb.vecs / np.linalg.norm(emb.vecs, axis=1, keepdims=True)      # cosine geometry
rows = []
for label, (a, b) in PAIRS.items():
    ca, cb = cid(*a), cid(*b)
    if ca is None or cb is None:
        rows.append((label, "not found")); continue
    ia, ib = G.pos([ca])[0], G.pos([cb])[0]
    rows.append((label, "pruned / not yet decided" if ia < 0 or ib < 0 else round(float(V[ia] @ V[ib]), 3)))
pd.DataFrame(rows, columns=["pair", "cosine (1 = same direction)"]).set_index("pair")

## 4. The embedding as a picture
**Sensible:** structure, not a blob; coloured by decade it should not be a simple left-to-right time gradient (if it is, try `MODE='cocite'` if you were on `sym`).

In [ ]:
meta = names(G.ids)
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].scatter(emb.vecs[:, 0], emb.vecs[:, 1], s=2, alpha=.4); ax[0].set_title("Fiedler (x) vs 3rd eigenvector (y)")
sc = ax[1].scatter(emb.vecs[:, 0], emb.vecs[:, 1], s=2, alpha=.5, c=meta.year.values, cmap="viridis"); ax[1].set_title("coloured by year decided")
plt.colorbar(sc, ax=ax[1]); plt.show()

## 5. Over time: embeddings per slice and subspace drift
**Sensible:** drift is moderate and roughly stable between quiet decades, larger across known reorganisations; the number of nodes grows monotonically.

In [ ]:
embs, graphs = {}, {}
for y in YEARS:
    t0 = time.time()
    s = st.at(y)
    ids_y = select(s, "cluster", where=f"d.dst_id = '{COURT}' AND n.precedential_status = 'Published'",
                   join="JOIN edge_cluster_in_docket_docket cd ON cd.src_id = n.id JOIN edge_docket_filed_in_court d ON d.src_id = cd.dst_id")
    Gy = build(s, ("cluster", "cites", "cluster"), ids_y, mode=MODE, min_degree=MIN_DEG)
    if Gy.n < K + 2:
        print(y, "too small, skipped"); continue
    graphs[y] = Gy; embs[y] = Spectral(k=K)(Gy)
    print(f"{y}: {Gy.n:>7,} nodes  {Gy.W.nnz//2:>9,} edges  λ2..λ4 = {embs[y].vals[:3].round(4)}  gap->{embs[y].gap()}   ({time.time()-t0:.0f}s)")

In [ ]:
ys = sorted(embs)
k_cmp = 6                                             # compare the leading subspace, not all K
d = [align(embs[a], embs[b], k=k_cmp) for a, b in zip(ys, ys[1:])]
tab = pd.DataFrame({"from": ys[:-1], "to": ys[1:],
                    "common_nodes": [len(x.common) for x in d],
                    "grassmann": [round(x.grassmann, 3) for x in d],
                    "angles_deg": [np.degrees(x.angles).round(0).astype(int).tolist() for x in d]})
display(tab.set_index("from"))
plt.figure(figsize=(6,3)); plt.plot(ys[1:], tab.grassmann, "o-"); plt.title(f"subspace drift (k={k_cmp}), {COURT} {MODE}"); plt.xlabel("to year"); plt.show()

Reading the angles: small early angles with large late ones means the leading directions persist while finer ones reshuffle. All angles near 90° means the slices are unrelated — a bug or far too small a graph.

## 6. Side by side after Procrustes
**Sensible:** the same cases sit in the same place in both panels (that is what the rotation is for); new regions appear rather than everything moving.

In [ ]:
a, b = ys[-2], ys[-1]
A = align(embs[a], embs[b], k=k_cmp)
fig, ax = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
ax[0].scatter(A.reference[:, 0], A.reference[:, 1], s=2, alpha=.4); ax[0].set_title(f"{a}")
ax[1].scatter(A.rotated[:, 0],  A.rotated[:, 1],  s=2, alpha=.4); ax[1].set_title(f"{b}, rotated onto {a}")
for i in np.argsort(-graphs[a].degree[graphs[a].pos(A.common)])[:12]:          # label a dozen hubs in both
    nm = names([A.common[i]]).case_name.iloc[0][:25]
    ax[0].annotate(nm, A.reference[i, :2], fontsize=6); ax[1].annotate(nm, A.rotated[i, :2], fontsize=6)
plt.show()

## 7. Window instead of cumulative
**Sensible:** the window graph is smaller and its spectrum coarser; the drift between windows is larger than between cumulative slices (windows forget, cumulative graphs remember).

In [ ]:
wins = {}
for y0, y1 in zip(YEARS[:-1], YEARS[1:]):
    s = st.window(y0, y1)                                    # citations FORMED by decisions filed in [y0, y1)
    ids_w = select(s, "cluster", where=f"d.dst_id = '{COURT}' AND n.precedential_status = 'Published'",
                   join="JOIN edge_cluster_in_docket_docket cd ON cd.src_id = n.id JOIN edge_docket_filed_in_court d ON d.src_id = cd.dst_id")
    # window co-citation among SCOTUS cases, cited together by SCOTUS decisions filed in [y0, y1)
    # (same construction as the cumulative slices, so the drift numbers are comparable)
    Gw = build(s, ("cluster", "cites", "cluster"), ids_w, mode=MODE, min_degree=MIN_DEG)
    if Gw.n < K + 2: continue
    wins[y1] = Spectral(k=K)(Gw)
    print(f"[{y0},{y1}): {Gw.n:>7,} nodes  λ2 = {wins[y1].vals[0]:.4f}")
wy = sorted(wins)
wd = [align(wins[a], wins[b], k=k_cmp) for a, b in zip(wy, wy[1:])]
display(pd.DataFrame({"from": wy[:-1], "to": wy[1:], "common_nodes": [len(x.common) for x in wd],
                      "grassmann": [round(x.grassmann, 3) for x in wd],
                      "angles_deg": [np.degrees(x.angles).round(0).astype(int).tolist() for x in wd]}).set_index("from"))
print(f"(max possible grassmann at k={k_cmp} is {np.sqrt(k_cmp)*np.pi/2:.2f} = every direction at 90°)")